# Real-Time Hub-to-Hub ETA Prediction with Kafka and Spark Structured Streaming

**HyperLogistics-Stream**

**Course.** COMP-SCI 5540 Principles of Big Data Management, Fall 2026, University of Missouri-Kansas City  
**Instructor.** Dr. Md Yusuf Sarwar Uddin  
**Authors.** Tony Nguyen (tmnc62@umsystem.edu) and Joe Doan (jdbc6@umsystem.edu)

## Abstract

Middle-mile freight hubs often learn how late an inbound trip is only when it arrives. HyperLogistics-Stream predicts each trip's arrival delay at departure time and serves the prediction in real time. We use US domestic flights from the BTS On-Time Performance dataset as a public proxy for scheduled hub-to-hub trips, fused with NOAA hourly surface weather at the destination. Kafka replays the recorded 2024 departures, Spark Structured Streaming enriches and scores each micro-batch with a Spark MLlib model, and the results land in a Parquet Medallion store behind a FastAPI service and a Streamlit dashboard. We evaluate prediction accuracy against two baselines (RQ1) and measure throughput and latency across a factorial sweep of Spark and Kafka settings (RQ2).

## Research questions

- **RQ1 (accuracy).** Do departure delay, streaming weather and lane history beat the "delay unchanged" baseline and the historical lane mean (MAE, RMSE)? The models are Spark MLlib Gradient-Boosted Trees and Random Forest. They train on 2018, 2019, 2021, 2022 and 2023, with 2020 excluded as a COVID anomaly. They are tested on the 2024 holdout, which is the same year the Kafka producer replays. Only features known at departure are used. Predicted ETA = scheduled arrival + departure delay + predicted change.
- **RQ2 (system performance).** How do cores (2/4/8), shuffle partitions (8/64/200), Kafka replay rate (1k/5k/20k events/s) and `maxOffsetsPerTrigger` (5k/20k/100k) change throughput, batch time and p50/p95 end-to-end latency?

## Datasets

- **BTS Reporting Carrier On-Time Performance.** One row per US domestic flight, with scheduled and actual times and delays. This run uses 2018, 2019 and 2021 to 2024. Source is https://www.transtats.bts.gov.
- **NOAA Integrated Surface Database (ISD).** Hourly temperature, visibility and wind per weather station, for the same years. Source is https://www.ncei.noaa.gov/data/global-hourly/.

Section 2 describes how the two are joined.

## Architecture

```text
OFFLINE TRAINING (Section 4)

  BTS 2018, 2019, 2021-2023 ---+
  BTS 2024 (holdout) ----------+--> pipeline/01_batch_train.py ------> models/ on Drive
  NOAA ISD hourly weather -----+    Spark MLlib GBT + RF               gbt_pipeline_v1
                                    MAE / RMSE on the 2024 holdout     lane_mean.parquet
                                                                              |
STREAMING (Section 5)                                                         |
                                                                              v
  BTS 2024 --> 02_kafka_producer.py --> Kafka topic -----> 03_stream_processor.py
               replay in WheelsOff      flights_stream     Spark Structured Streaming
               order at a fixed rate    (8 partitions)     one foreachBatch query:
                                                           parse, join NOAA 2024 weather
                                                           and lane mean, score with GBT
                                                                      |
            +-------------------+-------------------+-----------------+--+
            v                   v                   v                    v
         Bronze              Silver              Gold                 Gold
         flights_raw         flights_enriched    predictions          hub_congestion
         raw Kafka JSON      parsed + features   ETA, risk flag       sliding windows
            |                                       |                    |
            +------ Parquet Medallion store on Google Drive -------------+
                                                    |
SERVING (Section 6)                                 v
                              FastAPI (api/main.py, port 8000)
                              Streamlit (dashboard/app.py, port 8501)
```

The four course stages map onto this design. Kafka is the ingest stage. The Parquet Medallion store is the storage stage. Spark Structured Streaming and MLlib are the analysis stage. FastAPI and Streamlit are the reporting stage.

## Runtime requirements

- Colab Pro with the High-RAM machine shape. Use Runtime > Change runtime type and pick High-RAM. No GPU is needed.
- At least 8 vCPUs is recommended for the full RQ2 cores dimension. The sweep only includes core counts the VM actually has. Section 1.4 prints the detected vCPUs and RAM.
- Google Drive with roughly 30 GB free (estimate). Raw data, models and Parquet tables persist there across sessions.

## Sections

All runtimes are estimates. Actual times depend on the Colab VM and on whether the data is already on Drive.

| # | Section | What it does | Rough runtime |
| :--- | :--- | :--- | :--- |
| 1 | Setup | Java, packages, Drive, clone, parameters, Kafka | 3 to 5 min |
| 2 | Data | BTS and NOAA download, size inventory | 40 to 90 min on the first run, a few minutes once files are on Drive |
| 3 | EDA | January 2024 sample and five figures | 2 to 5 min |
| 4 | RQ1 | Batch training and accuracy results | 20 to 45 min |
| 5 | Streaming | Producer, stream job, Medallion check | 3 to 5 min |
| 6 | Serving | FastAPI and Streamlit with live responses | 1 to 2 min |
| 7 | RQ2 | Benchmark sweep, summary table, figures | about 30 min for the quick grid on 8 vCPUs, about 100 min for the full grid |
| 8 | Conclusions | Answers, limitations, future work | none |
| A | Appendix | Troubleshooting | none |

## How to run

1. Open the notebook in Colab and select the High-RAM runtime.
2. Start from a fresh runtime and use Runtime > Run all.
3. Approve the Google Drive popup in Section 1.2. Run all waits there until you do.
4. Keep the tab open. Colab disconnects idle sessions, and Sections 2, 4 and 7 run for a long time.

Every cell can be rerun. Downloads skip finished files. Kafka and the servers start only when they are not already up. The Medallion tables append on every streaming run, so their row counts grow across reruns.

## 1. Setup

The whole pipeline runs on one Colab VM. Kafka and ZooKeeper run as local processes, and Spark runs in local mode. Raw data, models, Parquet tables and benchmark results live on Google Drive, so they survive a runtime reset. Spark checkpoints stay on the local disk, because they do not need to persist.

### 1.1 Install Java 11 and Python packages

PySpark 3.5 needs a JVM, so the cell installs OpenJDK 11. The Python packages cover Spark, the Kafka client, the API, the dashboard and Parquet access.

In [ ]:
import subprocess
import sys

subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-11-jdk-headless"], check=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "pyspark==3.5.0",
        "confluent-kafka>=2.3.0",
        "fastapi>=0.111.0",
        "uvicorn[standard]>=0.29.0",
        "streamlit>=1.35.0",
        "pydeck>=0.9.0",
        "plotly>=5.20.0",
        "pyarrow>=14.0.0",
        "pyyaml>=6.0",
        "tqdm>=4.66.0",
        "tabulate>=0.9.0",
        "tzdata>=2024.1",
        "nest-asyncio>=1.6.0",
    ],
    check=True,
)

import pyspark

print(f"PySpark {pyspark.__version__} installed")

### 1.2 Mount Google Drive

Colab shows a popup asking for Drive access. Run all waits here until you approve it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### 1.3 Clone the pipeline branch

The cell clones `feat/tony-pipeline`, or fetches it and hard-resets an existing clone to the remote branch. A rerun therefore always runs the latest pushed code. The printed commit ties this notebook's outputs to one version of the code.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/JoeDoan/HyperLogistic_Stream.git"
BRANCH = "feat/tony-pipeline"
REPO_DIR = Path("/content/HyperLogistic_Stream")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
commit = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True, check=True).stdout.strip()
print(f"{BRANCH} at {commit}")

### 1.4 Parameters

This is the only cell that defines paths and run parameters. Later cells read from it and never redefine them.

- `HYPERLOGISTICS_CONFIG` is exported once. Every script launched afterwards inherits it and reads `config/pipeline_config_colab.yaml`.
- The training, test and stream years come from the `training` block of that config. Changing the experiment means editing the config, not the notebook.
- `DOWNLOAD_YEARS` is the union of those years.
- The cell checks that every data path in the config sits under `DRIVE_ROOT`, so outputs cannot silently land on the ephemeral VM disk.
- It prints the vCPU count and RAM and warns when the VM is below 24 GB RAM or 8 vCPUs.

The cell also defines the helpers later cells use. `run_logged` runs a script in the foreground, echoes its output and saves a full log under `/content/logs`. `start_background` launches a long-running process with its output in a log file. `wait_until` polls a readiness check and fails with the log tail if the process dies or times out.

In [ ]:
import os
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

import yaml

DRIVE_ROOT = Path("/content/drive/MyDrive/HyperLogistics")
CONFIG_PATH = REPO_DIR / "config" / "pipeline_config_colab.yaml"
FIGS_DIR = REPO_DIR / "reports" / "figures"
LOG_DIR = Path("/content/logs")
JAVA_HOME = Path("/usr/lib/jvm/java-11-openjdk-amd64")
KAFKA_VERSION, SCALA_VERSION = "3.7.0", "2.13"
KAFKA_DIR = Path(f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}")

os.environ["HYPERLOGISTICS_CONFIG"] = str(CONFIG_PATH)
os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PATH"] = f"{JAVA_HOME}/bin:{os.environ['PATH']}"
# Python children writing to a pipe block-buffer stdout, which would delay progress output by minutes.
os.environ["PYTHONUNBUFFERED"] = "1"

CFG = yaml.safe_load(CONFIG_PATH.read_text())
PATHS = {name: Path(p) for name, p in CFG["paths"].items()}
off_drive = [name for name, p in PATHS.items() if name != "checkpoints" and DRIVE_ROOT not in p.parents]
if off_drive:
    raise ValueError(f"Config paths {off_drive} are not under DRIVE_ROOT {DRIVE_ROOT}")

TRAIN_YEARS = CFG["training"]["train_years"]
TEST_YEAR = CFG["training"]["val_year"]
STREAM_YEAR = CFG["training"]["stream_year"]
DOWNLOAD_YEARS = sorted(set(TRAIN_YEARS) | {TEST_YEAR, STREAM_YEAR})
BOOTSTRAP = CFG["kafka"]["bootstrap_servers"]
TOPIC = CFG["kafka"]["topic"]

CPU_COUNT = os.cpu_count()
with open("/proc/meminfo") as f:
    RAM_GB = next(int(line.split()[1]) for line in f if line.startswith("MemTotal")) / 1024**2

FIGS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config          {CONFIG_PATH}")
print(f"Drive root      {DRIVE_ROOT}")
print(f"Train years     {TRAIN_YEARS}")
print(f"Test year       {TEST_YEAR}")
print(f"Stream year     {STREAM_YEAR}")
print(f"Download years  {DOWNLOAD_YEARS}")
print(f"vCPUs           {CPU_COUNT}")
print(f"RAM             {RAM_GB:.1f} GB")
if RAM_GB < 24:
    print(f"WARNING: {RAM_GB:.0f} GB RAM is below 24 GB. Select the High-RAM runtime or training may run out of memory.")
if CPU_COUNT < 8:
    print(f"WARNING: {CPU_COUNT} vCPUs is below 8. RQ2 will skip core counts above {CPU_COUNT}.")


def log_file(name):
    return LOG_DIR / f"{name}.log"


def tail(name, lines=20):
    return "\n".join(log_file(name).read_text(errors="replace").splitlines()[-lines:])


def run_logged(cmd, hide=()):
    name = Path(cmd[1]).stem
    start = time.time()
    with open(log_file(name), "wb") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for raw in proc.stdout:
            log.write(raw)
            line = raw.decode(errors="replace").rstrip("\r\n").split("\r")[-1]
            if not any(h in line for h in hide):
                print(line, flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    print(f"[{name}] finished in {(time.time() - start) / 60:.1f} min")


def start_background(name, cmd):
    return subprocess.Popen(cmd, stdout=open(log_file(name), "wb"), stderr=subprocess.STDOUT)


def port_open(port):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("localhost", port)) == 0


def http_ok(url):
    try:
        with urllib.request.urlopen(url, timeout=2) as response:
            return response.status == 200
    except OSError:
        return False


def wait_until(ready, timeout, name, proc):
    deadline = time.time() + timeout
    while not ready():
        if proc.poll() is not None:
            raise RuntimeError(f"{name} exited with code {proc.returncode}\n{tail(name)}")
        if time.time() > deadline:
            raise TimeoutError(f"{name} not ready after {timeout} s\n{tail(name)}")
        time.sleep(1)

### 1.5 Start Kafka

Kafka 3.7 runs from the Apache binary release with ZooKeeper, without Docker. The cell is idempotent. It skips ZooKeeper or the broker when port 2181 or 9092 is already open. Otherwise it starts the process and polls the port until it accepts connections, failing after 60 s. The topic is created with `--if-not-exists`, so a rerun leaves it as it is. The partition count comes from `kafka.partitions` in the config, which is 8.

In [ ]:
KAFKA_URL = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/{KAFKA_DIR.name}.tgz"

if not KAFKA_DIR.exists():
    subprocess.run(f"wget -qO- {KAFKA_URL} | tar -xz -C {KAFKA_DIR.parent}", shell=True, check=True)

server_props = KAFKA_DIR / "config" / "server.properties"
if "# HyperLogistics-Stream" not in server_props.read_text():
    with open(server_props, "a") as f:
        f.write(
            "\n# HyperLogistics-Stream\n"
            "message.max.bytes=10485760\n"
            "log.retention.hours=2\n"
            "num.partitions=8\n"
            "auto.create.topics.enable=true\n"
        )

for name, port, script, props in [
    ("zookeeper", 2181, "zookeeper-server-start.sh", "zookeeper.properties"),
    ("kafka", 9092, "kafka-server-start.sh", "server.properties"),
]:
    if port_open(port):
        print(f"{name} already listening on port {port}")
        continue
    proc = start_background(name, [str(KAFKA_DIR / "bin" / script), str(KAFKA_DIR / "config" / props)])
    wait_until(lambda: port_open(port), 60, name, proc)
    print(f"{name} listening on port {port}")

kafka_topics = str(KAFKA_DIR / "bin" / "kafka-topics.sh")
subprocess.run(
    [
        kafka_topics, "--bootstrap-server", BOOTSTRAP,
        "--create", "--if-not-exists",
        "--topic", TOPIC,
        "--partitions", str(CFG["kafka"]["partitions"]),
        "--replication-factor", "1",
    ],
    check=True,
)
describe = subprocess.run(
    [kafka_topics, "--bootstrap-server", BOOTSTRAP, "--describe", "--topic", TOPIC],
    capture_output=True, text=True, check=True,
)
print(describe.stdout)

## 2. Data

**BTS Reporting Carrier On-Time Performance.** The Bureau of Transportation Statistics publishes one row per domestic flight operated by a reporting carrier. Each row has the scheduled and actual departure and arrival times, the wheels-off time, departure and arrival delay, distance, and cancellation and diversion flags. The data comes as one zip archive per month.

**NOAA Integrated Surface Database.** ISD holds hourly surface observations per weather station. We use wind speed, visibility and temperature, parsed from the `WND`, `VIS` and `TMP` fields. `scripts/download_noaa.py` fetches one CSV per station and year for 40 major US airports from NOAA's public AWS mirror.

**Airport to station mapping.** Each airport's ICAO code, such as `KATL`, is matched to an active station in NOAA's `isd-history.csv` inventory. Local BTS times are converted to UTC with each airport's time zone. Flights then join weather on the destination station and the UTC hour of scheduled arrival. The BTS weather-delay field is not used, because it is known only after arrival and covers extreme weather only.

**Why flights.** Middle-mile trucking telematics are proprietary. Scheduled flights between hubs are a public proxy with the same structure. Each trip has a fixed lane, a schedule, weather exposure and congestion at the destination. Section 8 lists what the proxy misses.

**Scale.** The course requires at least 5 GB of raw data. Section 2.3 measures the downloaded size and checks it against that minimum.

### 2.1 Download BTS flights

The cell downloads all 12 months of every year in `DOWNLOAD_YEARS` and extracts each archive to CSV with `--extract`. Both download scripts skip files that are already complete, so a rerun with Drive mounted is fast.

In [ ]:
run_logged(
    [
        sys.executable, "scripts/download_bts.py",
        "--years", *map(str, DOWNLOAD_YEARS),
        "--output-dir", str(PATHS["raw_bts"]),
        "--extract",
    ]
)

### 2.2 Download NOAA weather

The cell downloads hourly weather for the same years. It first maps the 40 airports to stations, then fetches one CSV per station and year.

In [ ]:
run_logged(
    [
        sys.executable, "scripts/download_noaa.py",
        "--years", *map(str, DOWNLOAD_YEARS),
        "--output-dir", str(PATHS["raw_noaa"]),
    ]
)

### 2.3 Data inventory

The table counts the extracted CSVs per source and year and sums their size. Only CSVs for `DOWNLOAD_YEARS` count, not the zip archives. The last line checks the total against the 5 GB course minimum.

In [ ]:
import re

import pandas as pd

BTS_MONTH_FILE = re.compile(r"_(\d{4})_\d{1,2}\.csv$")

records = []
for f in PATHS["raw_bts"].glob("*.csv"):
    match = BTS_MONTH_FILE.search(f.name)
    if match and int(match.group(1)) in DOWNLOAD_YEARS:
        records.append(("BTS", int(match.group(1)), f.stat().st_size))
for year in DOWNLOAD_YEARS:
    for f in (PATHS["raw_noaa"] / str(year)).glob("*.csv"):
        records.append(("NOAA", year, f.stat().st_size))

inventory = (
    pd.DataFrame(records, columns=["source", "year", "bytes"])
    .groupby(["source", "year"])
    .agg(csv_files=("bytes", "size"), gb=("bytes", lambda b: b.sum() / 1e9))
    .reset_index()
)
display(inventory.round(2))

total_gb = inventory["gb"].sum()
print(f"Total raw CSV size {total_gb:.2f} GB")
print(f"{'PASS' if total_gb >= 5 else 'FAIL'} against the 5 GB course minimum")

> **TODO(results):** Record the total raw size in GB and the BTS and NOAA shares from the table above. Use these numbers in the report's dataset section.

## 3. Exploratory data analysis

EDA runs on a sample, not on the full dataset. It takes the first 100,000 rows of the January 2024 BTS file and keeps non-cancelled, non-diverted flights between the 24 hub airports listed in `scripts/run_eda.py`. It then converts local times to UTC and joins destination weather in pandas. The figures motivate the feature choices in Section 4.

### 3.1 Build the sample

In [ ]:
import sys

from IPython.display import Image

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from scripts.run_eda import clean_bts_data, clean_noaa_weather, generate_visualizations, perform_stream_join

bts_csv = PATHS["raw_bts"] / f"On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_{TEST_YEAR}_1.csv"
flights = clean_bts_data(str(bts_csv), max_rows=100000)
weather = clean_noaa_weather(str(PATHS["raw_noaa"] / str(TEST_YEAR)))
joined = perform_stream_join(flights, weather)
generate_visualizations(joined, str(FIGS_DIR))
print(f"Joined sample has {len(joined):,} flights")

### 3.2 Delay distributions

Departure delay, arrival delay, and their difference `delta_delay`, which is the RQ1 label.

In [ ]:
display(Image(filename=str(FIGS_DIR / "fig1_delay_distributions.png"), width=900))

> **TODO(results):** Describe the skew of departure and arrival delay and where `delta_delay` is centered. Say what that implies for Baseline 1.

### 3.3 Destination wind speed

`delta_delay` by destination surface wind speed, in knot bins.

In [ ]:
display(Image(filename=str(FIGS_DIR / "fig2_wind_speed_vs_delay.png"), width=900))

> **TODO(results):** Say whether `delta_delay` rises with destination wind speed, and from which bin.

### 3.4 Destination visibility

Mean arrival delay and `delta_delay` by destination visibility class.

In [ ]:
display(Image(filename=str(FIGS_DIR / "fig3_visibility_vs_delay.png"), width=900))

> **TODO(results):** Compare low-visibility classes with clear conditions for both panels.

### 3.5 Hourly pattern

Flight volume and mean departure delay by scheduled departure hour in UTC.

In [ ]:
display(Image(filename=str(FIGS_DIR / "fig4_hourly_departure_patterns.png"), width=900))

> **TODO(results):** Name the peak hours for volume and for delay, and whether delay builds through the day.

### 3.6 Hub comparison

Inbound volume and the share of arrivals delayed more than 15 minutes at nine major hubs.

In [ ]:
display(Image(filename=str(FIGS_DIR / "fig5_hub_congestion_comparison.png"), width=900))

> **TODO(results):** Name the busiest hub and the hub with the highest delayed share. Say whether they are the same.

## 4. RQ1. Can departure-time features beat simple baselines?

### Method

**Label.** The model predicts `delta_delay = ArrDelay - DepDelay`, the minutes a flight gains or loses after it departs. The departure delay is already known at wheels-off, so predicting the change keeps it from dominating the error.

**Predicted ETA.** Predicted ETA = scheduled arrival + departure delay + predicted `delta_delay`. The stream job writes it to the Gold table as `predicted_eta_utc`.

**Features.** Only values known at departure are used. The list is `features.numeric` in the config, and 4.1 prints it.

| Feature | Meaning |
| :--- | :--- |
| `DepDelay` | Departure delay in minutes |
| `Distance` | Lane distance in miles |
| `DepHourUTC` | UTC hour of wheels-off |
| `ArrHourUTC` | UTC hour of scheduled arrival |
| `DayOfWeek` | Day of week of the flight |
| `DestWindSpeed` | Destination wind speed in m/s |
| `DestVisibility` | Destination visibility in meters |
| `DestTemp` | Destination temperature in degrees Celsius |
| `LaneMeanDelay` | Mean `delta_delay` of the origin-destination lane in the training years |

The weather features have one caveat. They are the observation at the destination for the scheduled arrival hour. At departure a real system would only have a forecast, so the model sees perfect-forecast weather. Section 8 lists this as a limitation.

**Data.** Cancelled and diverted flights are removed. Both endpoints must be among the hub airports in the config.

**Baselines.** Baseline 1 predicts `delta_delay = 0`, so the departure delay carries through unchanged. Baseline 2 predicts the historical lane mean. The lane mean is computed from the training years only, so no holdout information leaks in.

**Models.** Spark MLlib Gradient-Boosted Trees and Random Forest. Hyperparameters come from the `training` block of the config.

**Split.** The models train on 2018, 2019, 2021, 2022 and 2023. 2020 is excluded as a COVID anomaly. The test set is the 2024 holdout, the same year the Kafka producer replays in Section 5.

**Metrics.** MAE and RMSE in minutes on the 2024 holdout.

### 4.1 Train

The cell runs `pipeline/01_batch_train.py` with 64 shuffle partitions. The script reads every training and test CSV with Spark, joins weather, builds the lane mean table, evaluates both baselines, and trains GBT and RF. It writes the models, `lane_mean.parquet` and `training_metrics.json` to Drive. Set `SKIP_RF = True` to train GBT only when memory or time is tight. The streaming job only needs GBT.

In [ ]:
SKIP_RF = False

print(f"Features     {CFG['features']['numeric']}")
print(f"Train years  {TRAIN_YEARS}")
print(f"Test year    {TEST_YEAR}")
run_logged(
    [sys.executable, "pipeline/01_batch_train.py", "--shuffle-partitions", "64"]
    + (["--skip-rf"] if SKIP_RF else [])
)

### 4.2 Results

The cell reads `training_metrics.json` and compares all four predictors on the holdout. The last column is the relative MAE reduction over Baseline 1. The figure is saved to `reports/figures/fig_rq1_model_comparison.png`. It has a second panel with GBT feature importances when the metrics file includes them.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

metrics = json.loads((PATHS["models"] / "training_metrics.json").read_text())
train_label = ", ".join(map(str, metrics["train_years"]))
test_label = str(metrics["val_year"])

print(f"Train years {train_label}: {metrics['train_count']:,} flights")
print(f"Test year {test_label}: {metrics['val_count']:,} flights")
print(f"Training wall time {metrics['total_elapsed_sec'] / 60:.1f} min")

PREDICTORS = [
    ("Baseline 1: delta = 0", "b1"),
    ("Baseline 2: historical lane mean", "b2"),
    ("Random Forest", "rf"),
    ("Gradient-Boosted Trees", "gbt"),
]
rq1_df = pd.DataFrame(
    [
        {"Model": name, "MAE (min)": metrics[f"{key}_mae"], "RMSE (min)": metrics[f"{key}_rmse"]}
        for name, key in PREDICTORS
        if metrics.get(f"{key}_mae") is not None
    ]
)
rq1_df["MAE improvement vs Baseline 1 (%)"] = 100 * (metrics["b1_mae"] - rq1_df["MAE (min)"]) / metrics["b1_mae"]
display(rq1_df.round(2))

fi = metrics.get("gbt_feature_importances") or {}
fig, axes = plt.subplots(1, 2 if fi else 1, figsize=(14 if fi else 8, 5), squeeze=False)
axes = axes[0]
x = range(len(rq1_df))
w = 0.35
axes[0].bar([i - w / 2 for i in x], rq1_df["MAE (min)"], width=w, label="MAE (min)", color="#2563eb")
axes[0].bar([i + w / 2 for i in x], rq1_df["RMSE (min)"], width=w, label="RMSE (min)", color="#dc2626")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(rq1_df["Model"], rotation=20, ha="right")
axes[0].set_ylabel("Error (minutes)")
axes[0].set_title(f"RQ1: {test_label} holdout error, trained on {train_label}", fontweight="bold")
axes[0].legend()

if fi:
    fi_s = pd.Series(fi).sort_values()
    axes[1].barh(fi_s.index, fi_s.values, color="#059669")
    axes[1].set_xlabel("Relative importance")
    axes[1].set_title("RQ1: GBT feature importances", fontweight="bold")

plt.tight_layout()
out_path = FIGS_DIR / "fig_rq1_model_comparison.png"
plt.savefig(out_path, dpi=300)
plt.show()
print(f"Saved {out_path}")

> **TODO(results):** Report GBT and RF MAE and RMSE against both baselines, the percent MAE improvement over Baseline 1, and the top 3 GBT feature importances. State in one sentence whether the models beat both baselines.

## 5. Streaming pipeline

This section runs the online half of the system on the 2024 replay.

**Producer.** `pipeline/02_kafka_producer.py` loads 2024 BTS flights, sorts them by wheels-off time and emits one JSON event per departure to `flights_stream` at a fixed rate. Each event carries its emit timestamp. The stream job uses that timestamp to measure end-to-end latency.

**Stream job.** `pipeline/03_stream_processor.py` runs a single Structured Streaming query with `foreachBatch`. Each micro-batch is parsed, broadcast-joined with static 2024 NOAA weather and the lane mean table, scored with the GBT model, and written to all four Medallion tables. One query instead of four means Kafka is read once per batch and every layer sees the same rows.

**Medallion layers.** All four tables are Parquet on Drive. Paths come from the `paths` block of the config.

| Layer | Config key | Contents |
| :--- | :--- | :--- |
| Bronze | `bronze` | Raw Kafka records with offset, partition, timestamp and JSON payload |
| Silver | `silver` | Parsed events with UTC timestamps, joined weather and lane features |
| Gold predictions | `gold_predictions` | Predicted `delta_delay`, predicted ETA and a high-risk flag per flight |
| Gold hub congestion | `gold_congestion` | Inbound count, mean predicted delta and high-risk count per destination and sliding arrival window |

A flight is high risk when its predicted delta exceeds `streaming.high_risk_threshold_min`. Window length and slide come from the `streaming` block of the config.

### 5.1 Start the producer

The producer starts in the background at 1,000 events/s and loops over the first 200,000 events. The cell waits until the producer logs that it is ready to emit. The stream job reads from the latest offset, so it must not start before events flow. Producer output goes to `/content/logs/producer_demo.log`.

In [ ]:
RATE, LIMIT = 1000, 200000

if "producer_proc" in globals() and producer_proc.poll() is None:
    producer_proc.kill()
    producer_proc.wait()

producer_proc = start_background(
    "producer_demo",
    [
        sys.executable, "pipeline/02_kafka_producer.py",
        "--rate", str(RATE),
        "--limit", str(LIMIT),
        "--bootstrap", BOOTSTRAP,
        "--loop",
    ],
)
wait_until(
    lambda: "Ready to emit" in log_file("producer_demo").read_text(errors="replace"),
    300, "producer_demo", producer_proc,
)
print(tail("producer_demo", lines=3))

### 5.2 Run the stream job

The stream job runs in `--test` mode, which stops after 4 micro-batches or 60 s. It uses `min(4, CPU_COUNT)` cores, 8 shuffle partitions, at most 5,000 offsets per trigger and a 5 s trigger. Its batch metrics are labelled `demo_stream`. The producer is stopped when the job ends, including when it fails.

In [ ]:
CORES = min(4, CPU_COUNT)

try:
    run_logged(
        [
            sys.executable, "pipeline/03_stream_processor.py",
            "--cores", str(CORES),
            "--shuffle-partitions", "8",
            "--max-offsets", "5000",
            "--trigger-interval", "5",
            "--bootstrap", BOOTSTRAP,
            "--benchmark-label", "demo_stream",
            "--test",
        ]
    )
finally:
    producer_proc.kill()
    producer_proc.wait()
    print("Producer stopped")

### 5.3 Medallion check

The cell counts the rows of each layer with `pyarrow.dataset`, which reads Parquet footers instead of loading the data. It then shows 5 rows of the Gold predictions table. Counts include earlier runs, because every streaming run appends.

In [ ]:
import pandas as pd
import pyarrow.dataset as pads

LAYERS = {
    "Bronze": "bronze",
    "Silver": "silver",
    "Gold predictions": "gold_predictions",
    "Gold hub congestion": "gold_congestion",
}


def layer_dataset(key):
    return pads.dataset(str(PATHS[key]), format="parquet", partitioning="hive")


layer_rows = []
for layer, key in LAYERS.items():
    dataset = layer_dataset(key)
    layer_rows.append({"layer": layer, "files": len(dataset.files), "rows": dataset.count_rows(), "path": str(PATHS[key])})
display(pd.DataFrame(layer_rows))
display(layer_dataset("gold_predictions").head(5).to_pandas())

> **TODO(results):** Record the row count of each layer. Confirm that Bronze, Silver and Gold predictions match, since every batch writes the same rows to all three. Explain one sample prediction in a sentence.

## 6. Serving

FastAPI exposes the Gold tables as REST endpoints, and Streamlit renders a live operations dashboard. Both read the Parquet tables on Drive directly. They run as background processes on the Colab VM and are reached through Colab's built-in port proxy, which needs no account or token. The stream job stopped at the end of 5.2, so both show the data it wrote.

### 6.1 Start FastAPI and Streamlit

Each server starts only if its health URL does not already answer. The cell then polls the health URL until it returns HTTP 200, failing after 90 s. Logs go to `/content/logs/api.log` and `/content/logs/dashboard.log`.

In [ ]:
API_PORT, DASH_PORT = 8000, 8501

SERVICES = {
    "api": {
        "health": f"http://localhost:{API_PORT}/",
        "cmd": [sys.executable, "-m", "uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", str(API_PORT)],
    },
    "dashboard": {
        "health": f"http://localhost:{DASH_PORT}/_stcore/health",
        "cmd": [
            sys.executable, "-m", "streamlit", "run", "dashboard/app.py",
            "--server.port", str(DASH_PORT),
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
        ],
    },
}

for name, service in SERVICES.items():
    if http_ok(service["health"]):
        print(f"{name} already serving at {service['health']}")
        continue
    proc = start_background(name, service["cmd"])
    wait_until(lambda: http_ok(service["health"]), 90, name, proc)
    print(f"{name} serving at {service['health']}")

### 6.2 Query the API

The cell calls three endpoints and prints their JSON. The hub congestion endpoint has no limit parameter, so the cell prints its first 3 windows.

In [ ]:
import json


def get_json(path):
    try:
        with urllib.request.urlopen(f"http://localhost:{API_PORT}{path}", timeout=30) as response:
            return response.status, json.load(response)
    except urllib.error.HTTPError as error:
        return error.code, json.load(error)


for path in ["/api/v1/system/metrics", "/api/v1/flights/active?limit=3", "/api/v1/hubs/ATL/congestion"]:
    status, body = get_json(path)
    print(f"GET {path}  HTTP {status}")
    print(json.dumps(body[:3] if isinstance(body, list) else body, indent=2))

### 6.3 Open the API docs and dashboard

Colab maps each port to a private URL. It opens only in this browser session for the notebook owner and stops working when the runtime ends. A saved copy of this notebook shows the links but cannot open them.

In [ ]:
from google.colab.output import eval_js

api_url = eval_js(f"google.colab.kernel.proxyPort({API_PORT})").rstrip("/")
dashboard_url = eval_js(f"google.colab.kernel.proxyPort({DASH_PORT})")
print(f"FastAPI docs         {api_url}/docs")
print(f"Streamlit dashboard  {dashboard_url}")

## 7. RQ2. How do Spark and Kafka settings change throughput and latency?

### Method

**Factors.** The sweep varies four settings. They are Spark cores, `spark.sql.shuffle.partitions`, the Kafka replay rate and `maxOffsetsPerTrigger`. The full grid in the config has 3 levels of each.

**Procedure.** `benchmark/run_benchmark.py` runs every combination. For each configuration it starts a fresh producer at the target rate and waits until it emits. It then launches the stream job in a fresh Spark process with `local[N]` in `--test` mode. A fresh process per configuration keeps JVM warmup, caches and checkpoints from carrying over between runs.

**Metrics.**

- Throughput is rows processed per second of batch time.
- Batch time is the wall time of one `foreachBatch` call. The summary reports its p50 and p95 across batches.
- End-to-end latency is the time from an event's emit timestamp in the producer to the end of the batch that wrote it. Each batch records p50 and p95, and the summary reports the median of each across batches.

**Warmup.** The first batches of a run include JVM startup and the first Kafka fetch, so they are discarded. The runner drops the first `benchmark.warmup_batches` batches, or only the first batch when a run has too few. 7.1 prints the configured count.

**Cores.** The grid only includes core counts the VM has. On a 4-vCPU VM, `local[8]` would oversubscribe the CPU and measure contention instead of scaling.

**Grid size.** With `QUICK = True` the sweep uses 2 levels for shuffle partitions, rate and offsets. With `QUICK = False` it uses the full lists from the `benchmark` block of the config. Each configuration takes about 75 s, which is an estimate.

### 7.1 Run the sweep

Metric files from earlier runs are moved to an `archive/` subfolder first. The analysis script reads every `metrics_*.jsonl` in the results folder. Without the move, files from another grid or from the 5.2 demo run would mix into the figures. Producer lines are kept out of the cell output and saved in `/content/logs/run_benchmark.log`.

In [ ]:
import shutil

QUICK = True

BENCH = CFG["benchmark"]
CORES_GRID = [c for c in (2, 4, 8) if c <= CPU_COUNT]
SHUFFLE_GRID = [8, 64] if QUICK else BENCH["shuffle_partitions"]
RATE_GRID = [1000, 5000] if QUICK else BENCH["ingestion_rates"]
OFFSETS_GRID = [5000, 20000] if QUICK else BENCH["max_offsets"]
n_configs = len(CORES_GRID) * len(SHUFFLE_GRID) * len(RATE_GRID) * len(OFFSETS_GRID)

print(f"Cores {CORES_GRID}  shuffle {SHUFFLE_GRID}  rates {RATE_GRID}  offsets {OFFSETS_GRID}")
print(f"{n_configs} configurations, {BENCH['warmup_batches']} warmup batches dropped per run")
print(f"Estimated runtime {n_configs * 75 / 60:.0f} min at about 75 s per configuration (estimate)")

results_dir = PATHS["benchmark_results"]
stale = sorted(results_dir.glob("metrics_*.jsonl"))
if stale:
    archive = results_dir / "archive" / time.strftime("%Y%m%dT%H%M%S")
    archive.mkdir(parents=True)
    for f in stale:
        shutil.move(f, archive / f.name)
    print(f"Archived {len(stale)} earlier metrics files to {archive}")

run_logged(
    [
        sys.executable, "benchmark/run_benchmark.py",
        "--cores", *map(str, CORES_GRID),
        "--shuffle", *map(str, SHUFFLE_GRID),
        "--rates", *map(str, RATE_GRID),
        "--offsets", *map(str, OFFSETS_GRID),
        "--bootstrap", BOOTSTRAP,
    ],
    hide=("[P-",),
)

### 7.2 Summary table

One row per configuration that produced metrics, sorted by throughput. The runner leaves out configurations that failed, so the first line compares the row count with the grid size.

In [ ]:
import pandas as pd

summary = pd.read_csv(PATHS["benchmark_results"] / "benchmark_summary.csv")
SUMMARY_COLUMNS = [
    "label", "cores", "shuffle_partitions", "ingestion_rate", "max_offsets", "batches_measured",
    "avg_throughput_rps", "p50_batch_ms", "p95_batch_ms", "median_p50_latency_ms", "median_p95_latency_ms",
]
print(f"{len(summary)} of {n_configs} configurations produced metrics")
display(summary.sort_values("avg_throughput_rps", ascending=False)[SUMMARY_COLUMNS].round(1))

### 7.3 Figures

The cell runs `benchmark/analyze_results.py` and shows the four RQ2 figures. The scalability curve needs at least two core counts, so it is missing on a 2-vCPU VM.

In [ ]:
from IPython.display import Image

run_logged([sys.executable, "benchmark/analyze_results.py"])

for name in [
    "fig_rq2_throughput_heatmap.png",
    "fig_rq2_latency_boxplot.png",
    "fig_rq2_scalability_curve.png",
    "fig_rq2_kafka_rate_impact.png",
]:
    path = FIGS_DIR / name
    if path.exists():
        display(Image(filename=str(path), width=950))
    else:
        print(f"{name} was not produced. See the analysis output above.")

> **TODO(results):** Interpret the four figures. Cover the best cores and shuffle combination in the heatmap, the spread of batch time and p95 latency in the boxplot, measured speedup against linear in the scalability curve, and where throughput stops tracking the replay rate.

## 8. Conclusions

### RQ1 answer

> **TODO(results):** Answer RQ1 in 2 to 3 sentences. Give the GBT MAE and RMSE on the 2024 holdout against Baseline 1 and Baseline 2, the percent MAE improvement over Baseline 1, whether GBT or RF did better, and which features mattered most.

### RQ2 answer

> **TODO(results):** Answer RQ2 in 3 to 4 sentences. Name the highest-throughput configuration, how throughput scaled with cores, the effect of shuffle partitions, how p95 latency changed with replay rate and `maxOffsetsPerTrigger`, and the vCPU count of the VM used.

### Limitations

- **Flights are a proxy for trucks.** BTS covers passenger carriers, not cargo carriers. Trucking adds Hours of Service limits, road traffic and dwell time at docks, and none of these appear in flight data. We found no study that validates flights as a trucking proxy.
- **Recorded replay.** The stream replays recorded 2024 flights instead of live feeds. The producer sets the event rate, not real traffic.
- **Perfect-forecast weather.** Destination weather is the observation at the scheduled arrival hour. A deployed system would only have a forecast at departure, so RQ1 overstates what the weather features can contribute.
- **Single node.** Spark runs in local mode on one Colab VM, and Kafka shares that machine. RQ2 measures scaling across cores, not across machines.
- **Cores bounded by Colab.** The VM's vCPU count caps the cores dimension. A VM with fewer than 8 vCPUs cannot test the 8-core level.
- **EDA sample.** Section 3 uses the first 100,000 rows of January 2024. Its figures show patterns, not estimates for the full dataset.

### Future work

- Add the previous-leg delay of the same tail number as a stateful streaming feature.
- RQ3 is a stretch goal that was not done. It is an LLM agent that answers dispatcher questions from the live prediction tables, kept off the streaming path.

## Appendix. Troubleshooting

**Kafka stops responding.** Colab can kill background processes after idle time. Rerun 1.5, which restarts whatever is down. If the broker still fails to start, stop both processes, clear their state and rerun 1.5. The topic is recreated.

```
!pkill -f kafka; rm -rf /tmp/kafka-logs /tmp/zookeeper
```

**Spark runs out of memory during training.** Set `SKIP_RF = True` in 4.1 to skip Random Forest. Or raise `spark.driver_memory` in `config/pipeline_config_colab.yaml` inside the clone and rerun from 1.4. Do not rerun 1.3 after editing, because it resets the clone and discards the edit. Commit the change to the branch to keep it.

**Drive mount fails because `/content/drive` already has files.** Something wrote to the mount point before mounting. Use Runtime > Disconnect and delete runtime, then Run all again.

**Colab disconnects while idle.** Long cells keep running only while the session is alive. Keep the tab open during Sections 2, 4 and 7. After a disconnect, run all again from a fresh runtime. Downloaded data, trained models and Parquet tables persist on Drive, so the downloads are not repeated.

**A script fails.** Each script's full output is in `/content/logs/<script name>.log`. Background processes log to `/content/logs/<process name>.log`.

**The stream job in 5.2 records no batches.** It reads from the latest Kafka offset, so it needs a running producer. Rerun 5.1, then 5.2.